In [ ]:
import mlflow
import os
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dataset = datasets.MNIST(root='../../datasets', train = True, download = False, transform=transforms.ToTensor())
test_dataset = datasets.MNIST(root = '../../datasets', train = False, download = False, transform=transforms.ToTensor())

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(nn.Conv2d(1, 32, 3, stride=1,padding=0), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2))
        self.layer2 = nn.Sequential(nn.Conv2d(32, 64, 3, stride=1,padding=0), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.output = nn.Sequential(nn.Flatten(), nn.Linear(64*5*5, 10))

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.output(x)

        return x

In [ ]:
mlflow.set_experiment("mnist-mlp")


with mlflow.start_run(run_name="cnn-2layers"):
    mlflow.log_param("shape-architecture", "(1,28,28)-(32,26,26)-(32,13,13)-(64,11,11)-(64,5,5)")
    mlflow.log_param("architecture", "Conv2d-BatchNorm2d-ReLU-Pool")
    mlflow.log_param("hidden_layers", 2)
    mlflow.log_param("activation", "ReLU")
    mlflow.log_param("optimizer", "Adam")
    mlflow.log_param("learning_rate", 0.001)

    model = Model().to(device)
    model.train()


    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

    loss_list = []

    for epoch in range(20):
        loss_sum = 0
        batch_num = 0
        
        for images, labels in train_loader:

            images, labels = images.to(device), labels.to(device)    

            batch_num += 1
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss_sum += loss.item()
            loss.backward()
            optimizer.step()
        
        
        print(f"({epoch}) : {loss_sum/batch_num}")
        mlflow.log_metric("train_loss", loss_sum/batch_num, step=epoch)
        loss_list.append(loss_sum/batch_num)
    
    model.eval()

    correct = 0
    samples = 0

    preds = []
    true = []

    for images, labels in test_loader:
        with torch.no_grad():

            images, labels = images.to(device), labels.to(device)    
            
            outputs = model(images)
            predicted = torch.argmax(outputs, dim=1)
            correct += (predicted == labels).sum().item()
            samples += labels.size(0)
            
            preds.extend(predicted.tolist())
            true.extend(labels.tolist())
            
    accuracy = correct/samples
    mlflow.log_metric("test_accuracy", accuracy)
    print(accuracy)

    plt.plot(loss_list)
    plt.title('Training Loss Over Time')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.show()



In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(true, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=range(10))
disp.plot()
plt.show()
